# Exploring Plaid Sandbox API Responses

This notebook walks through the Plaid API step by step so you can see exactly what each call does and what comes back. We'll:
1. Configure the Plaid client
2. Create a sandbox access token (shortcut — no browser needed)
3. Fetch accounts
4. Fetch transactions
5. Inspect the raw fields so we can validate our schema

## Step 1: Imports and config

`load_dotenv()` reads your `.env` file and makes the variables available via `os.environ`.
This keeps credentials out of the code.

In [13]:
import json
import os
from datetime import date, timedelta

from dotenv import load_dotenv
import plaid
from plaid.api import plaid_api
from plaid.model.sandbox_public_token_create_request import SandboxPublicTokenCreateRequest
from plaid.model.item_public_token_exchange_request import ItemPublicTokenExchangeRequest
from plaid.model.transactions_get_request import TransactionsGetRequest
from plaid.model.transactions_get_request_options import TransactionsGetRequestOptions
from plaid.model.products import Products

load_dotenv(dotenv_path="/home/bzou/dough-flow-db/.env")

# Verify credentials loaded
print("PLAID_CLIENT_ID:", os.environ.get("PLAID_CLIENT_ID", "NOT FOUND"))
print("PLAID_SECRET:   ", os.environ.get("PLAID_SECRET", "NOT FOUND")[:6] + "...")

PLAID_CLIENT_ID: 69c590c156628c000d6a366a
PLAID_SECRET:    c6279a...


## Step 2: Create the Plaid client

The client is the object you call all API methods on. It holds your credentials and knows which environment (sandbox vs production) to hit.

In [14]:
configuration = plaid.Configuration(
    host=plaid.Environment.Sandbox,
    api_key={
        "clientId": os.environ["PLAID_CLIENT_ID"],
        "secret":   os.environ["PLAID_SECRET"],
    },
)
client = plaid_api.PlaidApi(plaid.ApiClient(configuration))
print("Client created:", client)

Client created: <plaid.api.plaid_api.PlaidApi object at 0x7bce65cbb140>


## Step 3: Get a sandbox access token

Normally a user connects their bank through Plaid Link (a browser flow), which gives you a **public token**. You then exchange that for an **access token** to store and reuse.

Sandbox has a shortcut: you can create a public token directly via API, skipping the browser. We use `ins_109508` — "First Platypus Bank", Plaid's fake test institution.

In [15]:
pt_response = client.sandbox_public_token_create(
    SandboxPublicTokenCreateRequest(
        institution_id="ins_109508",
        initial_products=[Products("transactions")],
    )
)
public_token = pt_response["public_token"]
print("public_token:", public_token)

public_token: public-sandbox-553dd670-f175-48ef-8cb0-bfd32e9ef5f0


In [16]:
exchange_response = client.item_public_token_exchange(
    ItemPublicTokenExchangeRequest(public_token=public_token)
)
access_token = exchange_response["access_token"]
print("access_token:", access_token)

access_token: access-sandbox-2d982434-2fe4-40e3-9346-6a12164ad5fa


In [17]:
import time

# Plaid needs a few seconds to initialize the transactions product for a newly created sandbox item.
# Without this, transactions_get returns PRODUCT_NOT_READY immediately after token exchange.
print("Waiting for Plaid to initialize transactions product...")
time.sleep(10)
print("Done.")

Waiting for Plaid to initialize transactions product...
Done.


## Step 4: Fetch accounts

This shows us what an account object looks like — we want to confirm field names like `account_id`, `type`, `balances`, etc. map to what we have in our schema.

In [18]:
txn_response = client.transactions_get(
    TransactionsGetRequest(
        access_token=access_token,
        start_date=date.today() - timedelta(days=90),
        end_date=date.today(),
        options=TransactionsGetRequestOptions(count=5),
    )
)

print(f"Accounts returned: {len(txn_response['accounts'])}")
print(f"Transactions available: {txn_response['total_transactions']}\n")

# Inspect the first account
first_account = txn_response["accounts"][0].to_dict()
print("=== FIRST ACCOUNT (raw fields) ===")
print(json.dumps(first_account, indent=2, default=str))

Accounts returned: 12
Transactions available: 48

=== FIRST ACCOUNT (raw fields) ===
{
  "account_id": "gKWW8AVMxvSB1g9oVkraf931G433LDcEEy7a6",
  "balances": {
    "available": 100.0,
    "current": 110.0,
    "limit": null,
    "iso_currency_code": "USD",
    "unofficial_currency_code": null
  },
  "mask": "0000",
  "name": "Plaid Checking",
  "official_name": "Plaid Gold Standard 0% Interest Checking",
  "type": "depository",
  "subtype": "checking",
  "holder_category": "personal"
}


## Step 5: Inspect a transaction

This is the most important cell — we want to see every field on a transaction object and compare it against our `transactions` table schema.

In [19]:
first_txn = txn_response["transactions"][0].to_dict()
print("=== FIRST TRANSACTION (raw fields) ===")
print(json.dumps(first_txn, indent=2, default=str))

=== FIRST TRANSACTION (raw fields) ===
{
  "account_id": "8eEE3wGJbWivQaXDNeoRFk8Ebq886QSWWNJ41",
  "account_owner": null,
  "amount": 25.0,
  "authorized_date": "2026-04-13",
  "authorized_datetime": null,
  "category": null,
  "category_id": null,
  "check_number": null,
  "counterparties": [],
  "date": "2026-04-14",
  "datetime": null,
  "iso_currency_code": "USD",
  "location": {
    "address": null,
    "city": null,
    "region": null,
    "postal_code": null,
    "country": null,
    "lat": null,
    "lon": null,
    "store_number": null
  },
  "logo_url": null,
  "merchant_entity_id": null,
  "merchant_name": null,
  "name": "CREDIT CARD 3333 PAYMENT *//",
  "payment_channel": "other",
  "payment_meta": {
    "reference_number": null,
    "ppd_id": null,
    "payee": null,
    "by_order_of": null,
    "payer": null,
    "payment_method": null,
    "payment_processor": null,
    "reason": null
  },
  "pending": false,
  "pending_transaction_id": null,
  "personal_finance_catego

## Step 6: See all 5 transactions side by side

Look for patterns — what varies, what's null, what fields might be unreliable.

In [20]:
for i, txn in enumerate(txn_response["transactions"]):
    t = txn.to_dict()
    print(f"--- Transaction {i+1} ---")
    print(f"  transaction_id : {t.get('transaction_id')}")
    print(f"  account_id     : {t.get('account_id')}")
    print(f"  amount         : {t.get('amount')}")
    print(f"  date           : {t.get('date')}")
    print(f"  authorized_date: {t.get('authorized_date')}")
    print(f"  name           : {t.get('name')}")
    print(f"  merchant_name  : {t.get('merchant_name')}")
    print(f"  pending        : {t.get('pending')}")
    print(f"  category       : {t.get('category')}")
    print(f"  personal_finance_category: {t.get('personal_finance_category')}")
    print()

--- Transaction 1 ---
  transaction_id : BEbbQlxLBrtWQ5agDqJKSXNr3kPwBQf4ze9b4
  account_id     : 8eEE3wGJbWivQaXDNeoRFk8Ebq886QSWWNJ41
  amount         : 25.0
  date           : 2026-04-14
  authorized_date: 2026-04-13
  name           : CREDIT CARD 3333 PAYMENT *//
  merchant_name  : None
  pending        : False
  category       : None
  personal_finance_category: {'confidence_level': 'LOW', 'detailed': 'TRANSFER_OUT_ACCOUNT_TRANSFER', 'primary': 'TRANSFER_OUT', 'version': 'v2'}

--- Transaction 2 ---
  transaction_id : k3WWGb5MNJi9b7KavAwRIwlxzvebQMULJP4lv
  account_id     : gKWW8AVMxvSB1g9oVkraf931G433LDcEEy7a6
  amount         : 5.4
  date           : 2026-04-14
  authorized_date: 2026-04-13
  name           : Uber 063015 SF**POOL**
  merchant_name  : Uber
  pending        : False
  category       : None
  personal_finance_category: {'confidence_level': 'VERY_HIGH', 'detailed': 'TRANSPORTATION_TAXIS_AND_RIDE_SHARES', 'primary': 'TRANSPORTATION', 'version': 'v2'}

--- Transaction 

In [21]:
# Build a lookup from account_id -> account type/subtype
account_lookup = {
    a["account_id"]: {"type": a["type"], "subtype": a["subtype"]}
    for a in [acc.to_dict() for acc in txn_response["accounts"]]
}

print("=== TRANSACTIONS WITH ACCOUNT TYPE ===\n")
for i, txn in enumerate(txn_response["transactions"]):
    t = txn.to_dict()
    acct = account_lookup.get(t["account_id"], {})
    print(f"--- Transaction {i+1} ---")
    print(f"  name         : {t['name']}")
    print(f"  amount       : {t['amount']}")
    print(f"  account_type : {acct.get('type')} / {acct.get('subtype')}")
    print(f"  pfc_primary  : {t.get('personal_finance_category', {}).get('primary')}")
    print()

=== TRANSACTIONS WITH ACCOUNT TYPE ===

--- Transaction 1 ---
  name         : CREDIT CARD 3333 PAYMENT *//
  amount       : 25.0
  account_type : depository / savings
  pfc_primary  : TRANSFER_OUT

--- Transaction 2 ---
  name         : Uber 063015 SF**POOL**
  amount       : 5.4
  account_type : depository / checking
  pfc_primary  : TRANSPORTATION

--- Transaction 3 ---
  name         : ACH Electronic CreditGUSTO PAY 123456
  amount       : 5850.0
  account_type : depository / money market
  pfc_primary  : RENT_AND_UTILITIES

--- Transaction 4 ---
  name         : CD DEPOSIT .INITIAL.
  amount       : 1000.0
  account_type : depository / cd
  pfc_primary  : ENTERTAINMENT

--- Transaction 5 ---
  name         : United Airlines
  amount       : -500.0
  account_type : depository / checking
  pfc_primary  : TRAVEL

